In [44]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ieee-fraud-detection')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/ieee-fraud-detection


In [45]:
#load libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [46]:
#Load the Data
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity     = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


In [47]:
#Merge the Data
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")

In [48]:
#Separate Target and Features
y = train["isFraud"]
X = train.drop("isFraud", axis=1)
X_test = test.copy()


In [49]:
#Memory Optimization
def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
        else:
            df[col] = df[col].astype("category")
    return df

X = reduce_mem(X)
X_test = reduce_mem(X_test)


In [50]:
#compute numeric and categorical columns only for columns that exist in BOTH datasets
common_cols = X.columns.intersection(X_test.columns)

num_cols = X[common_cols].select_dtypes(include=["int", "float"]).columns
cat_cols = X[common_cols].select_dtypes(include=["object", "category"]).columns


In [51]:
#Find missing columns
missing_cols = [col for col in X.columns if col not in X_test.columns]


In [52]:
#Create a DataFrame of missing columns
missing_df = pd.DataFrame(
    np.nan,
    index=X_test.index,
    columns=missing_cols
)

In [53]:
#Add missing columns to X_test
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = np.nan


/tmp/ipykernel_58/4026060868.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[col] = np.nan
/tmp/ipykernel_58/4026060868.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[col] = np.nan
/tmp/ipykernel_58/4026060868.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[col] = np

In [54]:
#Add missing columns to X (rare but safe)
for col in X_test.columns:
    if col not in X.columns:
        X[col] = np.nan


/tmp/ipykernel_58/152054656.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[col] = np.nan
/tmp/ipykernel_58/152054656.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[col] = np.nan
/tmp/ipykernel_58/152054656.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[col] = np.nan
/tmp/ipykerne

In [55]:
num_cols = X.select_dtypes(include=["int", "float", "int32", "int64", "float32", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns


In [56]:
#Handle Missing Values

X[num_cols] = X[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

X[cat_cols] = X[cat_cols].fillna("missing")
X_test[cat_cols] = X_test[cat_cols].fillna("missing")


TypeError: Cannot setitem on a Categorical with a new category (-1), set the categories first

In [ ]:
#Encode Categorical Features
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))


In [ ]:
#Feature Engineering time features
X["day"] = X["TransactionDT"] // (24*60*60)
X["hour"] = (X["TransactionDT"] // 3600) % 24

X_test["day"] = X_test["TransactionDT"] // (24*60*60)
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24


In [ ]:
#amount features
X["log_amt"] = np.log1p(X["TransactionAmt"])
X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])


In [ ]:
#Frequency encoding
for col in ["card1", "card2", "addr1", "addr2"]:
    freq = X[col].value_counts()
    X[col+"_freq"] = X[col].map(freq)
    X_test[col+"_freq"] = X_test[col].map(freq)


In [ ]:
#Train a High‑Performance LightGBM Model
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

train_set = lgb.Dataset(X_train, y_train)
valid_set = lgb.Dataset(X_valid, y_valid)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 256,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=20000,
    early_stopping_rounds=500
)

preds_valid = model.predict(X_valid)
print("AUC:", roc_auc_score(y_valid, preds_valid))

In [ ]:
#Predict on Test Set
test_pred = model.predict(X_test)


In [ ]:
#Build Submission File
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_pred
})

submission.to_csv("submission.csv", index=False)
